# 40. LCS from the authors' code: published norm and BatchNorm

LCS (Nunez et al., WACV 2023) learns a line in weight space whose points
are networks of different widths: width 0.25 at one end, 1.00 at the other,
and every width between them by interpolation. Run here from the authors'
repository (apple/learning-compressible-subspaces at e6d3924) with
`third_party/patches/lcs.patch` applied; `third_party/patches/lcs_README.md`
lists every change. In structured mode only the norm layers are lines, the
convolutions are shared, so the stored model is 23.76M parameters, 0.2% more
than one ResNet-50.

| | GPU | what |
|---|---|---|
| lcs_l_bn | 1 | LCS with BatchNorm, recalibrated at each width on 20 batches: our protocol |
| lcs_l_in | 0 | LCS with its published instance norm, which needs no recalibration |

Both: CIFAR-100, CIFAR ResNet-50 (3x3 stem, no max-pool), method `lcs_l`,
100 epochs, batch 256, SGD nesterov lr 0.2 with the authors' per-epoch
warm-up and cosine, wd 5e-4, widths 0.25-1.00, read at the sixteen widths.

## More than one session

On a T4 the BatchNorm run takes about 9-13 hours and the instance-norm run
about twice that. Each run checkpoints every epoch. When the session ends,
Save Version, then in a new run attach that version's output (pick the
version by number, not Latest) and run again: the notebook finds the
checkpoints and both runs continue. Attach the CIFAR-100 dataset as well.

In [ ]:
# Fixed for this notebook.
REPO_URL = 'https://github.com/duyh80456-code/new-pruning.git'
REPO_BRANCH = 'nhan'
UPSTREAM = 'https://github.com/apple/learning-compressible-subspaces.git'
COMMIT = 'e6d3924368faccbdfd3d89c4a4735dba947275c9'
UPSTREAM_DIR = 'learning-compressible-subspaces'
PATCH = 'lcs.patch'
PIP = 'pyyaml'
# (label, gpu, extra flags)
BRANCHES = [('lcs_l_in', '0', ''), ('lcs_l_bn', '1', '--norm BN --recal_batches 20')]
WORKER_ENV = {'JOSLIM_WORKERS': '2', 'LCS_WORKERS': '2'}

In [ ]:
import os, queue, re, shutil, subprocess, sys, threading, time
import torch

n_gpu = torch.cuda.device_count()
print('torch', torch.__version__, '| gpus', n_gpu)
if n_gpu == 0:
    raise SystemExit('No GPU. Set Accelerator to GPU T4 x2 and run again.')

WORK = '/kaggle/working'
OURS = os.path.join(WORK, 'new-pruning')
if not os.path.isdir(OURS):
    subprocess.run(['git', 'clone', '-b', REPO_BRANCH, REPO_URL, OURS],
                   check=True)
print('our code at', subprocess.run(
    ['git', '-C', OURS, 'log', '-1', '--format=%h %s'],
    capture_output=True, text=True).stdout.strip())

CODE = os.path.join(WORK, UPSTREAM_DIR)
if not os.path.isdir(CODE):
    subprocess.run(['git', 'clone', UPSTREAM, CODE], check=True)
    subprocess.run(['git', '-C', CODE, 'checkout', '-q', COMMIT], check=True)
    subprocess.run(['git', '-C', CODE, 'apply',
                    os.path.join(OURS, 'third_party', 'patches', PATCH)],
                   check=True)
print('upstream at', subprocess.run(
    ['git', '-C', CODE, 'log', '-1', '--format=%h %ad', '--date=short'],
    capture_output=True, text=True).stdout.strip(), '+', PATCH)

# install without letting pip replace Kaggle's torch
base = torch.__version__.split('+')[0]
with open(os.path.join(WORK, 'constraints.txt'), 'w') as handle:
    handle.write('torch=={}\n'.format(base))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-c',
                os.path.join(WORK, 'constraints.txt')] + PIP.split(),
               check=True)

In [ ]:
# the folder that contains cifar-100-python/, or a place to download it
DATA = None
for root, dirs, _ in os.walk('/kaggle/input', followlinks=True):
    if 'cifar-100-python' in dirs:
        DATA = root
        break
if DATA is None:
    DATA = os.path.join(WORK, 'data')
    os.makedirs(DATA, exist_ok=True)
    print('no CIFAR-100 attached: it will be downloaded into', DATA)
else:
    print('CIFAR-100 from', DATA)

In [ ]:
VAL_LINE = re.compile(
    r'val\s+([0-9.]+)\s+-1/\d+:\s+loss:\s+([0-9.eE+-]+),\s+'
    r'top1_error:\s+([0-9.]+)')
results = {}


def run_pinned(jobs):
    """one command per card, output tagged and interleaved"""
    lines = queue.Queue()
    procs = {}

    def pump(label, proc):
        for line in proc.stdout:
            lines.put((label, line.rstrip('\n')))
        proc.wait()
        lines.put((label, None))

    for label, gpu, command in jobs:
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=gpu, **WORKER_ENV)
        proc = subprocess.Popen(command, cwd=CODE, env=env, text=True,
                                stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT)
        procs[label] = proc
        threading.Thread(target=pump, args=(label, proc),
                         daemon=True).start()
        print('[{}] gpu {}: {}'.format(label, gpu, ' '.join(command)),
              flush=True)
    started, remaining = time.time(), len(jobs)
    while remaining:
        label, line = lines.get()
        if line is None:
            remaining -= 1
            print('[{}] exit code {} after {:.0f} min'.format(
                label, procs[label].returncode,
                (time.time() - started) / 60), flush=True)
            continue
        found = VAL_LINE.search(line)
        if found:
            width, loss, top1 = found.groups()
            results.setdefault(label, {})[round(float(width), 2)] = (
                float(loss), float(top1))
        print('[{}] {}'.format(label, line), flush=True)
    return {label: proc.returncode for label, proc in procs.items()}

In [ ]:
CK = os.path.join(WORK, 'ckpt')
for label, _, _ in BRANCHES:
    target = os.path.join(CK, label, 'last.pt')
    os.makedirs(os.path.dirname(target), exist_ok=True)
    if os.path.exists(target):
        continue
    for root, _, files in os.walk('/kaggle/input', followlinks=True):
        if 'last.pt' in files and os.path.basename(root) == label:
            shutil.copy(os.path.join(root, 'last.pt'), target)
            print('restored', label, 'from', root)
            break
    print(label, 'resumes from its checkpoint' if os.path.exists(target)
          else 'starts at epoch 1')

In [ ]:
WIDTHS = ','.join('{:.2f}'.format(0.25 + 0.05 * i) for i in range(16))
COMMON = ['--model', 'cresnet50', '--dataset', 'cifar100', '--method',
          'lcs_l', '--data_dir', DATA, '--epochs', '100',
          '--batch_size', '256', '--learning_rate', '0.2', '--momentum',
          '0.9', '--nesterov', '--weight_decay', '5e-4',
          '--width_factor_limits', '0.25,1.0', '--eval_width_factors',
          WIDTHS, '--skip_upstream_test']
# training ends with the sixteen protocol lines
codes = run_pinned([
    (label, gpu, [sys.executable, '-u', 'train_structured.py'] + COMMON
     + extra.split() + ['--save_dir', os.path.join(WORK, label),
                        '--ckpt_dir', os.path.join(CK, label),
                        '--log_prefix', label])
    for label, gpu, extra in BRANCHES])
print(codes)

## Results, against A (BX, 76.86)

Send back this table and the `val ... -1/100` lines above it.

In [ ]:
A_REF = {0.25: 75.26, 0.3: 75.35, 0.35: 75.54, 0.4: 76.01, 0.45: 76.3, 0.5: 76.77, 0.55: 76.63, 0.6: 77.03, 0.65: 77.13, 0.7: 77.2, 0.75: 77.39, 0.8: 77.56, 0.85: 77.6, 0.9: 77.99, 0.95: 78.01, 1.0: 78.04}
labels = [b[0] for b in BRANCHES]
print('{:>7}{:>9}'.format('width', 'A') + ''.join(
    '{:>12}{:>8}'.format(b[:11], 'vs A') for b in labels))
for width in sorted(A_REF):
    row = '{:>7.2f}{:>9.2f}'.format(width, A_REF[width])
    for label in labels:
        entry = results.get(label, {}).get(width)
        if entry is None:
            row += '{:>12}{:>8}'.format('-', '-')
            continue
        accuracy = 100.0 * (1.0 - entry[1])
        row += '{:>12.2f}{:>+8.2f}'.format(accuracy, accuracy - A_REF[width])
    print(row)
reference = sum(A_REF.values()) / len(A_REF)
print('\n{:24} mean {:.2f}'.format('A (BX)', reference))
for label in labels:
    table = results.get(label, {})
    if len(table) == len(A_REF):
        mean = sum(100.0 * (1.0 - v[1]) for v in table.values()) / len(table)
        print('{:24} mean {:.2f}   vs A {:+.2f}'.format(
            label, mean, mean - reference))
    else:
        print('{:24} {} of 16 widths read'.format(label, len(table)))

In [ ]:
# keep the checkpoints in the output, under a name the next session finds
print('checkpoints in', CK, sorted(os.listdir(CK)))